In [4]:
!pip install pandas chembl_webresource_client

  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 453.9 kB/s eta 0:00:000:00:01
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 982.7 kB/s eta 0:00:000:00:01
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 113.5 kB/s eta 0:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 205.8 kB/s eta 0:00:000:0100:01
Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86

In [5]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
target = new_client.target
target_query = target.search('EGFR')
targets = pd.DataFrame.from_dict(target_query)
targets[['target_chembl_id', 'pref_name', 'target_type', 'organism']]

,target_chembl_id,pref_name,target_type,organism
0,CHEMBL3608,Epidermal growth factor receptor,SINGLE PROTEIN,Mus musculus
1,CHEMBL4523747,EGFR/PPP1CA,PROTEIN-PROTEIN INTERACTION,Homo sapiens
2,CHEMBL5465557,CCN2-EGFR,PROTEIN-PROTEIN INTERACTION,Homo sapiens
3,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN,Homo sapiens
4,CHEMBL4523680,Protein cereblon/Epidermal growth factor receptor,PROTEIN-PROTEIN INTERACTION,Homo sapiens
5,CHEMBL2111431,Epidermal growth factor receptor and ErbB2 (HE...,PROTEIN FAMILY,Homo sapiens
6,CHEMBL2363049,Epidermal growth factor receptor,PROTEIN FAMILY,Homo sapiens
7,CHEMBL4523998,von Hippel-Lindau disease tumor suppressor/EGFR,PROTEIN-PROTEIN INTERACTION,Homo sapiens
8,CHEMBL4630723,ErbB-2/ErbB-3 heterodimer,PROTEIN COMPLEX,Homo sapiens
9,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,SINGLE PROTEIN,Homo sapiens


In [7]:
selected_target = 'CHEMBL203'
activity = new_client.activity
print("Downloading data for target:", selected_target)
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
df = pd.DataFrame.from_dict(res)
df.to_csv('../data/egfr_raw_data.csv', index=False)
df.head(3)

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,32260,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,NaN,0.041
1,None,NaN,32263,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,NaN,NaN,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,NaN,0.3
2,None,NaN,32265,[],CHEMBL615325,Inhibition of ligand-induced proliferation in ...,F,NaN,NaN,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,NaN,7.82


In [8]:
df_clean = df[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'standard_units']]
df_clean = df_clean.dropna(subset=['canonical_smiles', 'standard_value'])
df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'])
print(f"Number of molecules after cleaning: {len(df_clean)}")
df_clean.head()

Number of molecules after cleaning: 24344


,molecule_chembl_id,canonical_smiles,standard_value,standard_units
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,41.0,nM
1,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,nM
2,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,7820.0,nM
3,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,170.0,nM
4,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,40.0,nM


In [9]:
import numpy as np
df_clean = df_clean.groupby(['molecule_chembl_id', 'canonical_smiles']).mean(numeric_only=True).reset_index()
df_clean = df_clean[df_clean['standard_value'] > 0]
def calculate_pIC50(ic50):
    return 9 - np.log10(ic50)
df_clean['pIC50'] = df_clean['standard_value'].apply(calculate_pIC50)
df_clean.to_csv('../data/egfr_pIC50_data.csv', index=False)
print(f"Done. Final dataset contains {len(df_clean)} unique molecules.")
df_clean.head()

Done. Final dataset contains 13369 unique molecules.


,molecule_chembl_id,canonical_smiles,standard_value,pIC50
0,CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,64500.000000,4.190440
1,CHEMBL100714,COc1ccc(Nc2ncnc3cc(OC)c(OC)cc23)cc1OC,2800.000000,5.552842
2,CHEMBL1009,N[C@@H](Cc1ccc(O)c(O)c1)C(=O)O,451440.000000,3.345400
3,CHEMBL101253,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6819.233333,5.166264
4,CHEMBL101581,COc1cc2nccc(Oc3cccc(Br)c3)c2cc1OC,2500.000000,5.602060
